# Google Play Store Apps — Exploratory Data Analysis

**Python • Pandas • Matplotlib • Plotly**

Analysis of a historical (~2018) Google Play Store sample, exploring app listings, categories, monetization, app size, install thresholds and star ratings. Source: [Kaggle dataset](https://www.kaggle.com/datasets/lava18/google-play-store-apps).

## 1. Load and inspect the data
Review column types, missing values, and repeated names before cleaning.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display
import sys

ROOT = Path.cwd()
if not (ROOT/'data/raw').exists() and (ROOT.parent/'data/raw').exists():
    ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
source = ROOT/'data/raw/googleplaystore.csv'
if not source.is_file():
    raise FileNotFoundError('Download the source file first: python download_data.py')
raw = pd.read_csv(source)
print('Raw dataset shape:',raw.shape)
display(raw.head())

In [ ]:
display(raw.info())
display(raw.isna().sum().sort_values(ascending=False))
print('Exact duplicate rows:', raw.duplicated().sum())
print('Repeated app names:', raw['App'].duplicated().sum())

## 2. Clean and transform
Convert `Size`, `Installs`, and `Price` into numerical features. Remove malformed records and repeated app names while preserving missing star ratings.

In [ ]:
from src.cleaning import clean_google_play, parse_size_mb

clean, audit = clean_google_play(raw)
print('Cleaning audit:')
display(pd.Series(audit).to_frame('value'))
display(clean[['App','Category','Rating','Reviews_num','Size_MB','Installs_lower_bound','Price_usd']].head())

In [ ]:
examples = pd.DataFrame({
    'Raw size': ['20M', '512k', 'Varies with device', 'bad entry'],
})
examples['Converted MB'] = examples['Raw size'].map(parse_size_mb)
display(examples)

print('Missing after cleaning:')
display(clean[['Rating','Size_MB','Reviews_num','Installs_lower_bound','Price_usd']].isna().sum())

## 3. Explore categories, monetization, and content ratings

In [ ]:
category_count = clean['Category'].value_counts().head(10)
display(category_count.to_frame('apps'))
print('Most common category:', category_count.index[0])

In [ ]:
type_count = clean['Type'].value_counts()
share = (type_count / len(clean) * 100).round(1)
display(pd.DataFrame({'apps': type_count, 'percent': share}))

In [ ]:
display(clean['Content Rating'].value_counts().head(10))

## 4. Compare ratings and installation thresholds
Require at least 30 rated apps for category-level mean ratings. Installation values are lower bounds, not exact counts.

In [ ]:
category_ratings = (
    clean.groupby('Category')
         .agg(apps=('App','count'), rated_apps=('Rating','count'), mean_rating=('Rating','mean'))
         .query('rated_apps >= 30')
         .sort_values('mean_rating', ascending=False)
)
display(category_ratings.head(10))

In [ ]:
display(clean.groupby('Type')['Installs_lower_bound'].agg(['count','median','mean']))
display(clean.groupby('Type')['Rating'].agg(['count','mean','median']))

## 5. Static visualizations with Matplotlib

In [ ]:
chart_dir = ROOT / 'charts'
chart_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 5))
category_count.sort_values().plot(kind='barh', ax=ax, color='#0f766e')
ax.set(title='Top 10 App Categories by Number of Listings', xlabel='Number of apps', ylabel='')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.savefig(chart_dir/'notebook_top_categories.png', dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
share.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90,
           colors=['#0f766e', '#e09f3e'])
ax.set(title='Share of Free and Paid Listings', ylabel='')
plt.tight_layout()
plt.savefig(chart_dir/'notebook_free_paid.png', dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
clean['Rating'].dropna().hist(ax=ax, bins=20, color='#7c3aed', edgecolor='white')
ax.set(title='Distribution of App Ratings', xlabel='Star rating (1–5)', ylabel='Number of rated apps')
plt.tight_layout()
plt.savefig(chart_dir/'notebook_ratings.png', dpi=160)
plt.show()

## 6. Interactive visualizations with Plotly
Observe associations between listed price, file size and ratings without inferring causation.

In [ ]:
size_data = clean.dropna(subset=['Size_MB','Rating'])
fig = px.scatter(size_data, x='Size_MB', y='Rating', color='Type',
                 hover_name='App', opacity=0.55,
                 title='App Size vs Star Rating',
                 labels={'Size_MB':'App size (MB)', 'Rating':'Star rating'})
fig.write_html(chart_dir/'size_vs_rating.html', include_plotlyjs=True)
fig.show()

In [ ]:
paid = clean[(clean['Type']=='Paid') & clean['Price_usd'].gt(0) & clean['Rating'].notna()]
fig = px.scatter(paid, x='Price_usd', y='Rating', hover_name='App', log_x=True,
                 title='Paid App Price vs Rating (logarithmic price axis)',
                 labels={'Price_usd':'App price (USD)','Rating':'Star rating'})
fig.write_html(chart_dir/'price_vs_rating.html', include_plotlyjs=True)
fig.show()

In [ ]:
fig = px.box(clean.dropna(subset=['Rating']), x='Type', y='Rating', color='Type',
             title='Free vs Paid Apps: Rating Distribution', points='outliers')
fig.write_html(chart_dir/'rating_boxplot.html', include_plotlyjs=True)
fig.show()

## 7. Analysis summary

In [ ]:
print('Apps analyzed:',f'{len(clean):,}')
print('Top listing category:',clean['Category'].value_counts().idxmax())
print('Free app share:',f"{clean['Type'].eq('Free').mean():.1%}")
print('Most common content rating:',clean['Content Rating'].value_counts().idxmax())
print('Mean rating among rated apps:',round(clean['Rating'].mean(),2))

## Limitations
- Historical non-exhaustive dataset (~2018), not a live view of the Play Store.
- App-name deduplication is an approximation; package IDs are unavailable.
- Installation brackets give **lower bounds**, not exact downloads or revenue.
- Ratings missing for some apps are excluded from rating statistics only.
- Relationships among size, price and star ratings are descriptive, not causal.

Run `python run_analysis.py` to produce the chart files in `charts/`.